In [2]:
import os 
os.chdir("D:/RAG-Based-Medical-Chatbox")

In [3]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [4]:
#Extract text from PDF files
def load_pdf(data):
    loader = DirectoryLoader(
        data, 
        glob="*.pdf", 
        loader_cls=PyPDFLoader
    ) 
    return loader.load()

extracted_data = load_pdf("data")

print("Number of documents loaded:", len(extracted_data))



Number of documents loaded: 637


In [5]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs = []
    for doc in docs:
        # Create a new Document with only the page_content and metadata
        src = doc.metadata.get("source")
        minimal_doc = Document(page_content=doc.page_content, metadata={"source": src})
        minimal_docs.append(minimal_doc)
    return minimal_docs

In [6]:
minimal_docs = filter_to_minimal_docs(extracted_data);
print(len(minimal_docs))
print(minimal_docs[1])

637
page_content='The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION' metadata={'source': 'data\\Medical_book.pdf'}


In [7]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
        length_function=len
    )

    split_docs = text_splitter.split_documents(minimal_docs)
    return split_docs

In [8]:
text_split_docs = text_split(minimal_docs)
print("Number of chunks:", len(text_split_docs))

Number of chunks: 5859


In [10]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5"
    )
    return embeddings

embeddings = download_embeddings()


In [11]:
vector = embeddings.embed_query("Hello world")
print(vector)
print(len(vector))

[0.015196080319583416, -0.022570615634322166, 0.008547102101147175, -0.07417059689760208, 0.003836381947621703, 0.0027135133277624846, -0.031267911195755005, 0.04463401436805725, 0.044055189937353134, -0.007871152833104134, -0.025200802832841873, -0.03336659446358681, 0.01442789752036333, 0.046538230031728745, 0.008555113337934017, -0.016145780682563782, 0.007405820768326521, -0.01901245303452015, -0.11472631245851517, -0.018157636746764183, 0.12635937333106995, 0.029702860862016678, 0.02528103068470955, -0.03421790525317192, -0.0409996472299099, 0.006617261562496424, 0.01027063000947237, 0.022362295538187027, 0.00443631736561656, -0.1273096203804016, -0.0161492470651865, -0.020380116999149323, 0.047212082892656326, 0.011579906567931175, 0.0681871771812439, 0.007298652082681656, -0.017852971330285072, 0.040782131254673004, -0.010269480757415295, 0.02375711500644684, 0.010602901689708233, -0.028584420680999756, 0.008159671910107136, -0.015180512331426144, 0.03089623525738716, -0.0659798

In [12]:
from dotenv import load_dotenv
import os
load_dotenv()
PINE_CONE_API_KEY = os.getenv("PINE_CONE_API_KEY")
os.environ["PINECONE_API_KEY"] = PINE_CONE_API_KEY

In [13]:
from pinecone import Pinecone
pc = Pinecone(api_key = PINE_CONE_API_KEY)

In [14]:
pc

In [16]:
index_name = "medical-chatbox"

In [ ]:
from pinecone import ServerlessSpec
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

CREATE AND LOAD INDEX from PINECONE

In [ ]:
from langchain_pinecone import PineconeVectorStore

# create and load the vector store
PineconeVectorStore.from_documents(
    documents=text_split_docs,
    embedding=embeddings,
    index_name = index_name
)

LOAD INDEX from EXISTING PINECONE INDEX

In [18]:
from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

ADD DOCUMENT TO EXISTING INDEX OF PINE CODE     

In [20]:
dswith = Document(
    page_content="Abdul Shakoor.",
    metadata={"source": "system"}
)

docsearch.add_documents (documents=[dswith])

['52e4c8f5-96ab-4cfd-8bdb-12ddb29bd2ba']

RETRIEVE DOCUMENT           

In [23]:
retriever = docsearch.as_retriever(search_type = "similarity" , search_kwargs={"k":3})
retrieved_doc = retriever.invoke("Abdul Shakoor")
retrieved_doc

[Document(id='52e4c8f5-96ab-4cfd-8bdb-12ddb29bd2ba', metadata={'source': 'system'}, page_content='Abdul Shakoor.'),
 Document(id='c4b991d0-6089-431e-acdc-5e929d666232', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Atkins has seen about 60,000 patients in his more\nthan 30 years of practice. He has also appeared on numer-\nous radio and television talk shows, has his own syndicat-\ned radio program, Your Health Choices , and authors the\nmonthly newsletter Dr. Atkins’ Health Revelations. Atkins\nhas received the World Organization of Alternative Medi-\ncine’s Recognition of Achievement Award and been\nnamed the National Health Federation’s Man of the Year.\nHe is director of the Atkins Center for Complementary'),
 Document(id='76aa9568-4bdc-4784-92ec-b436fa77117c', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Anxiety Disorders Association of America. 11900 Park Lawn\nDrive, Ste. 100, Rockville, MD 20852. (800) 545-7367.\n<http://www.adaa.org>.\nNational I